# Querying USGS data with DuckDB

This notebook demonstrates the optional
`dataretrieval.duckdb_connectors` package. There is one connector per
data source — `waterdata` for the OGC API, `wqp` for the
Water Quality Portal — and each wraps a [DuckDB](https://duckdb.org/)
connection so that endpoints can be queried directly from SQL.

## Why DuckDB?

The getters already return pandas DataFrames, so why involve another
engine? A few reasons:

* **Joining heterogeneous endpoints in one query.** Sites, daily
  values, time-series metadata and water-quality results live behind
  separate endpoints with different shapes. Once registered as views
  they can be combined with a single SQL statement — even across
  data sources.
* **Window functions and time aggregations.** Ranking the wettest
  months per gauge or computing rolling means is very compact in SQL.
* **Compose with files on disk.** DuckDB can read Parquet, CSV and
  remote S3 objects natively, so cached water data can be joined
  against external datasets without leaving SQL.

## Install

```bash
pip install dataretrieval[duckdb]
```

If `geopandas` is also installed, monitoring-location geometry is
converted to WKT plus `longitude`/`latitude` columns so the prototype
doesn't need the DuckDB spatial extension.

Set an [API token](https://api.waterdata.usgs.gov/signup/) for higher
rate limits:

```python
import os
os.environ["API_USGS_PAT"] = "your_api_key_here"
```

In [ ]:
from dataretrieval.duckdb_connectors import waterdata, wqp

wd = waterdata.connect()  # in-memory connection wired to the OGC waterdata API

## 1. Discover sites

Pull a small set of streamflow gauges in Illinois and register them as
the `sites` view. Every endpoint helper accepts `register_as=<name>`
to publish the result as a view that subsequent SQL can reference.

In [ ]:
wd.monitoring_locations(
    state_name="Illinois",
    site_type="Stream",
    register_as="sites",
    limit=200,
)
wd.sql(
    """
    SELECT monitoring_location_id, monitoring_location_name,
           county_name, drainage_area, longitude, latitude
    FROM sites
    WHERE drainage_area IS NOT NULL
    ORDER BY drainage_area DESC
    LIMIT 10
    """
)

## 2. Pull daily streamflow for a few gauges

Pick three Illinois River basin gauges and pull a year of daily mean
discharge. The waterdata getter accepts a list of monitoring-location
IDs and handles pagination internally, so the single call below
returns the entire year for all three sites in one DataFrame which we
register as the `daily` view.

In [ ]:
GAUGES = [
    "USGS-05586100",  # Illinois River at Valley City
    "USGS-05543500",  # Illinois River at Marseilles
    "USGS-05447500",  # Green River near Geneseo
]

wd.daily(
    monitoring_location_id=GAUGES,
    parameter_code="00060",   # discharge, cubic feet per second
    statistic_id="00003",     # daily mean
    time="2023-01-01/2023-12-31",
    register_as="daily",
)
wd.sql("SELECT count(*) AS n_rows, count(DISTINCT monitoring_location_id) AS n_sites FROM daily")

## 3. Aggregate: monthly mean flow per gauge

Once the data is in DuckDB, time-bucketing is a one-liner with
`date_trunc`. Compare with the equivalent pandas pipeline
(`groupby([id, pd.Grouper(...)])` and `unstack`) and the SQL is
noticeably more direct.

In [ ]:
wd.sql(
    """
    SELECT monitoring_location_id,
           date_trunc('month', time)::DATE AS month,
           round(avg(value), 1) AS mean_cfs,
           round(max(value), 1) AS max_cfs
    FROM daily
    GROUP BY monitoring_location_id, month
    ORDER BY monitoring_location_id, month
    """
)

## 4. Window function: top 5 highest-flow days per gauge

Window functions are where SQL really pays off. The query below ranks
every day within each gauge by discharge and returns the five
highest. The same operation in pandas is doable but reads less
cleanly.

In [ ]:
wd.sql(
    """
    SELECT *
    FROM (
        SELECT monitoring_location_id, time::DATE AS date, value,
               row_number() OVER (
                   PARTITION BY monitoring_location_id
                   ORDER BY value DESC
               ) AS rk
        FROM daily
    )
    WHERE rk <= 5
    ORDER BY monitoring_location_id, rk
    """
)

## 5. Join site metadata to daily values

`sites` and `daily` are independent endpoints; in SQL they're just
two views joined on `monitoring_location_id`. The query below produces
annual statistics enriched with site name and drainage area.

In [ ]:
# Re-pull sites filtered to just our gauges so we have their metadata
wd.monitoring_locations(
    monitoring_location_id=GAUGES,
    register_as="gauge_sites",
)

wd.sql(
    """
    SELECT s.monitoring_location_name AS name,
           s.drainage_area,
           round(avg(d.value), 1)  AS mean_cfs,
           round(min(d.value), 1)  AS min_cfs,
           round(max(d.value), 1)  AS max_cfs,
           round(avg(d.value) / NULLIF(s.drainage_area, 0), 3)
               AS unit_runoff_cfs_per_sqmi
    FROM gauge_sites s
    JOIN daily d USING (monitoring_location_id)
    GROUP BY s.monitoring_location_name, s.drainage_area
    ORDER BY mean_cfs DESC
    """
)

## 6. Latest readings across many sites

The `latest_continuous` endpoint returns one row per site — perfect
for a near-realtime dashboard.

In [ ]:
wd.latest_continuous(
    monitoring_location_id=GAUGES,
    parameter_code="00060",
    register_as="latest",
)
wd.sql(
    """
    SELECT s.monitoring_location_name AS name,
           l.time AS observed_at,
           l.value AS cfs,
           l.qualifier
    FROM gauge_sites s
    JOIN latest l USING (monitoring_location_id)
    ORDER BY observed_at DESC
    """
)

## 7. Cross-source: pair waterdata streamflow with WQP water quality

The Water Quality Portal (WQP) is a separate multi-agency repository
covering USGS plus EPA, state agencies, etc. Its connector follows
the same pattern, but the WQP API uses CamelCase parameters and a
different ID schema (e.g. `MonitoringLocationIdentifier`).

Below we open a second connection to WQP, pull water-quality results
for our Illinois River gauge, and ATTACH the waterdata in-memory
database into the WQP one — letting a single query join the two
sources.

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # WQP raises a UserWarning on legacy use

wq = wqp.connect(legacy=True)
wq.get_results(
    siteid="USGS-05586100",
    startDateLo="01-01-2010",
    startDateHi="12-31-2015",
    register_as="wqp_results",
)
wq.sql(
    """
    SELECT count(*) AS n_results,
           count(DISTINCT CharacteristicName) AS n_characteristics,
           min(ActivityStartDate) AS earliest,
           max(ActivityStartDate) AS latest
    FROM wqp_results
    """
)

In [ ]:
# Top 10 most-measured characteristics at this gauge in WQP
wq.sql(
    """
    SELECT CharacteristicName,
           count(*) AS n_obs
    FROM wqp_results
    WHERE ResultMeasureValue IS NOT NULL
    GROUP BY CharacteristicName
    ORDER BY n_obs DESC
    LIMIT 10
    """
)

### Joining across connections

Each connector owns its own duckdb connection. To join across the two,
we register one source's relation onto the other's connection. Below
we publish `gauge_sites` (from the waterdata connection) onto the WQP
connection and then JOIN it with `wqp_results`.

In [ ]:
# Pull just the metadata we want from waterdata as a DataFrame and
# register it on the WQP connection.
gauge_sites_df = wd.sql(
    """
    SELECT monitoring_location_id,
           monitoring_location_name AS site_name,
           drainage_area
    FROM gauge_sites
    """
).df()
wq.con.register("gauge_sites", gauge_sites_df)

wq.sql(
    """
    SELECT s.site_name,
           r.CharacteristicName AS characteristic,
           count(*) AS n_obs,
           round(avg(r.ResultMeasureValue), 3) AS mean_value
    FROM gauge_sites s
    JOIN wqp_results r
      ON s.monitoring_location_id = r.MonitoringLocationIdentifier
    WHERE r.ResultMeasureValue IS NOT NULL
      AND r.CharacteristicName IN ('pH', 'Temperature, water', 'Dissolved oxygen (DO)')
    GROUP BY s.site_name, r.CharacteristicName
    ORDER BY characteristic
    """
)

## 8. Drop into pandas / arrow when it's useful

Every helper returns a `duckdb.DuckDBPyRelation`. Convert to a
DataFrame for plotting, or to Arrow for downstream tooling, with the
standard DuckDB methods. The underlying connection is exposed as
`con.con` for any DuckDB feature the helpers don't cover (Parquet
export, attached databases, extensions).

In [ ]:
monthly = wd.sql(
    """
    SELECT date_trunc('month', time)::DATE AS month,
           monitoring_location_id,
           avg(value) AS mean_cfs
    FROM daily
    GROUP BY month, monitoring_location_id
    ORDER BY month
    """
).df()
monthly.head()

In [ ]:
# Optional: persist any joined result as Parquet for later reuse.
# wd.sql("COPY (SELECT * FROM daily) TO 'illinois_daily_2023.parquet'")

In [ ]:
wd.close()
wq.close()

## 9. Optional: native geometry via the spatial extension

By default, geometry is flattened to WKT plus `longitude`/`latitude`
columns so no extension is required. Pass `spatial=True` to
`connect()` to install + load DuckDB's `spatial` extension on the
underlying connection — `ST_GeomFromText`, `ST_DWithin_Sphere`, etc.
then become available against the registered views.

Install the compound extra to also pull in `geopandas` for richer
client-side geometry:

```bash
pip install dataretrieval[spatial]
```

(The DuckDB spatial extension itself is a runtime C++ extension
downloaded by DuckDB on first install; it's not a pip package.)

In [ ]:
wd_geo = waterdata.connect(spatial=True)
wd_geo.monitoring_locations(
    state_name="Illinois",
    site_type="Stream",
    register_as="sites",
    limit=200,
)

# Point-in-polygon: streamflow gauges within a central-Illinois box.
wd_geo.sql(
    """
    WITH central_il AS (
        SELECT ST_GeomFromText(
            'POLYGON((-90.5 39.5, -88.5 39.5, -88.5 40.5, -90.5 40.5, -90.5 39.5))'
        ) AS poly
    )
    SELECT monitoring_location_name, longitude, latitude
    FROM sites, central_il
    WHERE ST_Within(ST_GeomFromText(geometry), poly)
    ORDER BY monitoring_location_name
    LIMIT 10
    """
)

In [ ]:
wd_geo.close()

## Notes on the prototype

* The connectors are intentionally thin — they don't replace any of
  the underlying APIs, they just make results queryable as views.
  Pagination and CQL filter chunking are still handled inside
  `dataretrieval.waterdata`.
* By default, geometry is flattened to WKT so the demo doesn't
  require any extension. Pass `spatial=True` to `connect()` for
  native DuckDB geometry support (see section 9).
* `register_table(name, fn, **kwargs)` accepts any function that
  returns `(DataFrame, metadata)` — useful for endpoints not yet
  exposed as named helpers (e.g. `dataretrieval.waterdata.get_samples`).
* The legacy import path `from dataretrieval import duckdb_connector`
  is preserved as an alias for the waterdata connector.